# RNA-seq on Google Cloud Platform (Beginner Tutorial)
This notebook demonstrates a small end-to-end RNA-seq workflow on GCP using public data and low-resource tools.


## Overview of RNA-seq workflow
1. Create a modest Compute Engine VM (<=8 GB RAM)
2. Install FastQC + Salmon
3. Download small public FASTQ files
4. Download Ensembl reference transcriptome + GTF
5. QC with FastQC
6. Quantify with Salmon
7. Aggregate transcript output to gene-level table
8. Upload results to GCS


## 1) Configure user variables


In [ ]:
%%bash
set -euo pipefail
export PROJECT_ID="YOUR_GCP_PROJECT_ID"
export REGION="us-central1"
export ZONE="us-central1-a"
export VM_NAME="rna-seq-tutorial-vm"
export MACHINE_TYPE="e2-standard-2"   # 2 vCPU, 8 GB RAM
export BOOT_DISK_GB="50"
export BUCKET_NAME="YOUR_UNIQUE_BUCKET_NAME"


## 2) Create Compute Engine VM (run from Cloud Shell/local terminal)


In [ ]:
%%bash
set -euo pipefail
gcloud config set project "$PROJECT_ID"
gsutil mb -p "$PROJECT_ID" -l "$REGION" "gs://${BUCKET_NAME}" || true
gcloud compute instances create "$VM_NAME"   --zone "$ZONE"   --machine-type "$MACHINE_TYPE"   --boot-disk-size "$BOOT_DISK_GB"   --image-family ubuntu-2204-lts   --image-project ubuntu-os-cloud   --scopes cloud-platform


In [ ]:
%%bash
set -euo pipefail
gcloud compute ssh "$VM_NAME" --zone "$ZONE"


## 3) Install tools on the VM


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update
sudo apt-get install -y wget curl unzip pigz fastqc seqtk python3-pip
if ! command -v micromamba >/dev/null 2>&1; then
  "${SHELL}" <(curl -L micro.mamba.pm/install.sh)
fi
export MAMBA_ROOT_PREFIX="$HOME/micromamba"
source "$HOME/.bashrc" || true
micromamba create -y -n rnaseq_tutorial -c bioconda -c conda-forge salmon
python3 -m pip install --upgrade pip pandas


## 4) Create folders and verify tools


In [ ]:
%%bash
set -euo pipefail
mkdir -p "$HOME/rna_seq_tutorial"/{data,ref,qc,quant,results}
cd "$HOME/rna_seq_tutorial"
export MAMBA_ROOT_PREFIX="$HOME/micromamba"
eval "$(micromamba shell hook -s bash)"
micromamba activate rnaseq_tutorial
salmon --version
fastqc --version


## 5) Download small public RNA-seq FASTQ files (ENA)


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial/data"
export FASTQ_R1_URL="https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR390/005/SRR3907285/SRR3907285_1.fastq.gz"
export FASTQ_R2_URL="https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR390/005/SRR3907285/SRR3907285_2.fastq.gz"
wget -O sample_R1.fastq.gz "$FASTQ_R1_URL"
wget -O sample_R2.fastq.gz "$FASTQ_R2_URL"
ls -lh sample_R*.fastq.gz


### Optional downsampling for fast runtime


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial/data"
zcat sample_R1.fastq.gz | head -n 800000 | gzip > sample_sub_R1.fastq.gz
zcat sample_R2.fastq.gz | head -n 800000 | gzip > sample_sub_R2.fastq.gz
ls -lh sample_sub_R*.fastq.gz


## 6) Download matching Ensembl reference transcriptome + annotation


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial/ref"
export ENSEMBL_REL="110"
wget -O Homo_sapiens.GRCh38.cdna.all.fa.gz   "https://ftp.ensembl.org/pub/release-${ENSEMBL_REL}/fasta/homo_sapiens/cdna/Homo_sapiens.GRCh38.cdna.all.fa.gz"
wget -O Homo_sapiens.GRCh38.${ENSEMBL_REL}.gtf.gz   "https://ftp.ensembl.org/pub/release-${ENSEMBL_REL}/gtf/homo_sapiens/Homo_sapiens.GRCh38.${ENSEMBL_REL}.gtf.gz"
ls -lh *.gz


## 7) Build Salmon index


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial/ref"
export MAMBA_ROOT_PREFIX="$HOME/micromamba"
eval "$(micromamba shell hook -s bash)"
micromamba activate rnaseq_tutorial
salmon index -t Homo_sapiens.GRCh38.cdna.all.fa.gz -i salmon_index -k 31


## 8) Quality control with FastQC


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial"
fastqc -t 2 -o qc data/sample_sub_R1.fastq.gz data/sample_sub_R2.fastq.gz
ls -lh qc


## 9) Quantify with Salmon


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial"
export MAMBA_ROOT_PREFIX="$HOME/micromamba"
eval "$(micromamba shell hook -s bash)"
micromamba activate rnaseq_tutorial
salmon quant   -i ref/salmon_index -l A   -1 data/sample_sub_R1.fastq.gz -2 data/sample_sub_R2.fastq.gz   -p 2 --validateMappings   -o quant/sample_salmon
ls -lh quant/sample_salmon


## 10) Build a simple gene expression output table


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial"
python3 - <<'PYSCRIPT'
import gzip, re, pandas as pd
q = pd.read_csv('quant/sample_salmon/quant.sf', sep='	')
q['tx_id'] = q['Name'].str.replace(r'\..*$', '', regex=True)
rows=[]
with gzip.open('ref/Homo_sapiens.GRCh38.110.gtf.gz', 'rt') as fh:
    for line in fh:
        if line.startswith('#'): continue
        p=line.rstrip('
').split('	')
        if len(p) < 9 or p[2] != 'transcript': continue
        attrs=p[8]
        def ga(k):
            m=re.search(fr'{k} "([^"]+)"', attrs)
            return m.group(1) if m else None
        tx, gene, gname = ga('transcript_id'), ga('gene_id'), ga('gene_name')
        if tx and gene:
            rows.append((tx.split('.')[0], gene.split('.')[0], gname or gene))
map_df = pd.DataFrame(rows, columns=['tx_id','gene_id','gene_name']).drop_duplicates()
merged = q.merge(map_df, on='tx_id', how='left')
out = (merged.groupby(['gene_id','gene_name'], dropna=False, as_index=False)
       .agg(TPM=('TPM','sum'), NumReads=('NumReads','sum'))
       .sort_values('TPM', ascending=False))
out.to_csv('results/gene_expression_table.tsv', sep='	', index=False)
print(out.head(10).to_string(index=False))
print('Wrote results/gene_expression_table.tsv')
PYSCRIPT


## 11) Upload outputs to GCS


In [ ]:
%%bash
set -euo pipefail
cd "$HOME/rna_seq_tutorial"
export BUCKET_NAME="YOUR_UNIQUE_BUCKET_NAME"
export RUN_TAG="rna_seq_tutorial_$(date +%Y%m%d_%H%M%S)"
gsutil -m cp -r qc quant results "gs://${BUCKET_NAME}/${RUN_TAG}/"
gsutil ls -r "gs://${BUCKET_NAME}/${RUN_TAG}/**"


## 12) Next steps
- Run multiple samples and merge sample-level gene tables
- Add MultiQC summary
- Try HISAT2 + featureCounts for alignment-based counting
- Use DESeq2/edgeR for differential expression with biological replicates
